# Visualize Registration

In [ ]:
import ast
import cv2
import copy
import os
import sys
import time
import pandas as pd
import numpy as np
import open3d as o3d
from open3d.camera import PinholeCameraIntrinsic as o3dPinholeCameraIntrinsic
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))
from camera import Camera
from config import RegistrationConfig, load_config_yaml
from dataset_loader import DatasetLoader
from evaluation import Evaluation, compare_evaluations
from mask import masked_image
from paths import PathManager
from pose_initializer import PoseInitializer
from segmentation import predict_segment
from template import TemplateLoader

Set item and idx to the dataset you want to visualize. \
Experiment were executed over 300 <= idx <= 500.

In [ ]:
item = "apple"
idx = 300
path_to_data_csv = f"../results/evaluate_full_vs_partial_templates/{item}/results_combined.csv"
config = "../configs/baseline.yaml"
cam_name = "D435i"

cfg = load_config_yaml(config)

pm = PathManager(item)
cam = Camera.from_yaml(pm.get_camera_path(), cam_name)
loader = TemplateLoader(pm)
templates = loader.load_all()

dl = DatasetLoader(item)

In [ ]:
df = pd.read_csv(path_to_data_csv)

In [ ]:
def _scene_from_depth(cam, depth_img):
    fx, fy, cx, cy = cam.get_camera_intrinsics()
    intrinsic = o3dPinholeCameraIntrinsic(
        width=cam.width, height=cam.height,
        fx=fx, fy=fy, cx=cx, cy=cy
    )
    return o3d.geometry.PointCloud.create_from_depth_image(
        o3d.geometry.Image(depth_img),
        intrinsic=intrinsic, extrinsic=np.eye(4)
    )

In [ ]:
def _show_image(image, title, convert_to_bgr=False):
    if convert_to_bgr:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    plt.imshow(image)
    plt.title(title)
    plt.show()

In [ ]:
rgb, depth, t_gt = dl(idx)
try:
    pts, seg_img = predict_segment(rgb, item)
except TargetNotFoundError as e:
    print(f"[WARNING] [idx {idx}] {e}")
except SegmentationError as e:
    print(f"[ERROR] [idx {idx}] Segmentation failed: {e}")
masked_depth = masked_image(pts, depth)
scene = _scene_from_depth(cam, masked_depth)
full_scene = _scene_from_depth(cam, depth)

In [ ]:
_show_image(rgb, "RGB-Image", convert_to_bgr=True)
_show_image(seg_img, "Segmentation-Image", convert_to_bgr=True)
_show_image(depth, "Depth-Image")
_show_image(masked_depth, "Masked Depth-Image")

In [ ]:
RED = [1, 0, 0]
GREEN = [0, 1, 0]
BLUE = [0, 0.2, 1]
# GRAY = [0.3, 0.3, 0.3]
GRAY = [0.5, 0.5, 0.5]
BLACK = [0, 0, 0]

In [ ]:
def _to_T(val):
    """Convert transformation (list or stringified list) to 4x4 numpy array."""
    if isinstance(val, str):
        val = ast.literal_eval(val)
    return np.array(val, dtype=float)

def get_full_and_best_partial(df, idx):
    # Alle Zeilen für diesen Datensatz
    rows = df.loc[df["dataset_id"] == idx]
    if rows.empty:
        raise ValueError(f"No rows found for dataset_id == {idx}.")

    # --- Full model
    full_row = rows.loc[rows["template"] == "full_model"]
    T_full = None
    if not full_row.empty:
        T_full = _to_T(full_row.iloc[0]["transformation"])

    # --- Bestes Teiltemplate (höchste Fitness, excl. full_model)
    partials = rows.loc[rows["template"] != "full_model"]
    T_best_partial = None
    best_name = None
    if not partials.empty:
        best_partial_row = partials.loc[partials["fitness"].idxmax()]
        T_best_partial = _to_T(best_partial_row["transformation"])
        best_name = best_partial_row["template"]

    return T_full, T_best_partial, best_name

In [ ]:
T_full, T_best_partial, best_name = get_full_and_best_partial(df, idx)
print(T_full)
print(T_best_partial)
print(best_name)

In [ ]:
scene.paint_uniform_color(GREEN)
full_scene.paint_uniform_color(BLACK)
model = copy.deepcopy(next(t for t in templates if t.name == "full_model").pcd)
model.transform(t_gt)
model.paint_uniform_color(GRAY)


In [ ]:
if T_full is not None:
    full_template = copy.deepcopy(next(t for t in templates if t.name == "full_model").pcd)
    full_template.transform(T_full)
    full_template.paint_uniform_color(RED)

if T_best_partial is not None:
    partial_template = copy.deepcopy(next(t for t in templates if t.name == best_name).pcd)
    partial_template.transform(T_best_partial)
    partial_template.paint_uniform_color(BLUE)

In [ ]:
o3d.visualization.draw_geometries([full_scene, scene, model, full_template, partial_template], window_name=f"idx:{idx}, full_model and {best_name}")